In [1]:
#yolov9_here
from ultralytics import YOLO
import torch
import numpy as np
import time

In [2]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.12.1+cu126
True
NVIDIA GeForce RTX 4070 Ti SUPER


In [4]:
model = YOLO("yolov9s.pt")

In [5]:
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from ultralytics.utils.downloads import download


def visdrone2yolo(root: Path, split: str):
    root = Path(root)

    img_dir = root / split / "images"
    ann_dir = root / split / "annotations"
    label_dir = root / split / "labels"
    label_dir.mkdir(parents=True, exist_ok=True)

    def convert_box(size, box):
        dw = 1.0 / size[0]
        dh = 1.0 / size[1]
        return (
            (box[0] + box[2] / 2) * dw,
            (box[1] + box[3] / 2) * dh,
            box[2] * dw,
            box[3] * dh
        )

    for ann_file in tqdm(list(ann_dir.glob("*.txt")), desc=f"Converting {split}"):

        img_file = img_dir / ann_file.name.replace(".txt", ".jpg")

        if not img_file.exists():
            continue

        w, h = Image.open(img_file).size
        lines = []

        with open(ann_file, "r") as f:
            for row in f.read().strip().splitlines():
                row = row.split(",")

                if row[4] == "0":
                    continue

                cls = int(row[5]) - 1
                box = convert_box((w, h), tuple(map(int, row[:4])))

                lines.append(f"{cls} {box[0]} {box[1]} {box[2]} {box[3]}\n")

        with open(label_dir / ann_file.name, "w") as f:
            f.writelines(lines)




root = Path(r"D:/cv/Dataset")

urls = [
    "https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-train.zip",
    "https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-val.zip",
    "https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-test-dev.zip",
]

download(urls, dir=root, curl=True, threads=4)

for split in [
    "VisDrone2019-DET-train",
    "VisDrone2019-DET-val",
    "VisDrone2019-DET-test-dev"
]:
    visdrone2yolo(root, split)

WARNING Skipping D:\cv\Dataset\VisDrone2019-DET-val.zip unzip as destination directory D:\cv\Dataset\VisDrone2019-DET-val is not empty.
WARNING Skipping D:\cv\Dataset\VisDrone2019-DET-test-dev.zip unzip as destination directory D:\cv\Dataset\VisDrone2019-DET-test-dev is not empty.
WARNING Skipping D:\cv\Dataset\VisDrone2019-DET-train.zip unzip as destination directory D:\cv\Dataset\VisDrone2019-DET-train is not empty.


Converting VisDrone2019-DET-test-dev: 100%|██████████| 1610/1610 [00:24<00:00, 64.68it/s] 


In [6]:
start_time = time.time()
results = model.train(
    data= "D:\cv\Dataset\VisDrone.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    patience=50,
    cos_lr=True,
    max_det=500
    )
end_time = time.time()

<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\Ray\AppData\Local\Temp\ipykernel_28300\2388002663.py:3: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  data= "D:\cv\Dataset\VisDrone.yaml",


New https://pypi.org/project/ultralytics/8.4.148 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.84  Python-3.14.6 torch-2.12.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\cv\Dataset\VisDrone.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=500, mixup=0.0, mode=train, model=yolov9s.pt, momentum=0.937, mosaic=1.0,

In [7]:

metrics = model.val(
    data=r"D:\cv\Dataset\VisDrone.yaml",
    split="val",
    plots=True,
    save_json=True
)


Ultralytics 8.4.84  Python-3.14.6 torch-2.12.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
YOLOv9s summary (fused): 197 layers, 7,170,958 parameters, 0 gradients, 26.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2065.5613.5 MB/s, size: 126.7 KB)
val: Scanning D:\cv\Dataset\VisDrone2019-DET-val\labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 191.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 8.1it/s 4.3s<0.1s
                   all        548      38759      0.537      0.409        0.4      0.232
            pedestrian        520       8844      0.543      0.433      0.429      0.186
                people        482       5125      0.584      0.323      0.337      0.126
               bicycle        364       1287      0.313      0.179      0.148      0.064
                   car        515      14064      0.726      0.788      0.785      0.542
       

In [8]:
cm = metrics.confusion_matrix.matrix

FP = cm[:-1, -1].sum()
FN = cm[-1, :-1].sum()

TP = np.diag(cm[:-1, :-1]).sum()
training_time = end_time - start_time
print(training_time)
print(f"mAP@0.5       : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95  : {metrics.box.map:.4f}")
print(f"Precision     : {metrics.box.mp:.4f}")
print(f"Recall        : {metrics.box.mr:.4f}")
print("TP =", int(TP))
print("FP =", int(FP))
print("FN =", int(FN))


10983.434162378311
mAP@0.5       : 0.3998
mAP@0.5:0.95  : 0.2323
Precision     : 0.5373
Recall        : 0.4094
TP = 19630
FP = 6627
FN = 16120


In [3]:
model_best = YOLO(r"D:\cv\models\yolo9\runs\detect\train\weights\best.pt")

In [4]:
test_metrics = model_best.val(
    data=r"D:\cv\Dataset\VisDrone.yaml",
    split="test",
    plots=True,
    save_json=True
)

Ultralytics 8.4.84  Python-3.14.6 torch-2.12.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)
YOLOv9s summary (fused): 197 layers, 7,170,958 parameters, 0 gradients, 26.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2350.9956.9 MB/s, size: 177.5 KB)
val: Scanning D:\cv\Dataset\VisDrone2019-DET-test-dev\labels.cache... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 397.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 9.6it/s 10.5s<0.1s
                   all       1610      75102      0.456      0.364      0.328      0.187
            pedestrian       1197      21006      0.435      0.301      0.271      0.105
                people        797       6376      0.478      0.138      0.144     0.0488
               bicycle        377       1302      0.266      0.144      0.117     0.0468
                   car       1530      28074      0.675      0.748      0.723      0.

In [5]:
cm = test_metrics.confusion_matrix.matrix

FP = cm[:-1, -1].sum()
FN = cm[-1, :-1].sum()

TP = np.diag(cm[:-1, :-1]).sum()


print(f"mAP@0.5       : {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95  : {test_metrics.box.map:.4f}")
print(f"Precision     : {test_metrics.box.mp:.4f}")
print(f"Recall        : {test_metrics.box.mr:.4f}")
print("TP =", int(TP))
print("FP =", int(FP))
print("FN =", int(FN))

mAP@0.5       : 0.3285
mAP@0.5:0.95  : 0.1871
Precision     : 0.4557
Recall        : 0.3637
TP = 33458
FP = 13468
FN = 35495
